# Reviewer 1 — Paired Block-Based Significance Tests

**Why not naive tests:** FCN3, GraphCast and HRES are scored on the same times,
stations and lead times, so their errors are paired, not independent samples (rules
out Mann-Whitney U). And station-hours aren't independent either — a single storm
inflates errors at all 12 stations at once, and consecutive hours at one station are
autocorrelated — so treating every station-hour as one independent observation
understates uncertainty (pseudo-replication), and treating the 12 stations as the only
independent unit (n=12) is conservative but throws away all temporal structure. Every
test below instead resamples or aggregates at the **block level**: a block is a run of
calendar days, all 12 stations and all lead times pooled together, so cross-station and
serial dependence within a block is preserved rather than assumed away. The metric is
aggregated exactly as in the station-level cluster bootstrap: RMSE/Bias per (station, lead
time), averaged over lead times, then over the 12 stations (not pooled).

**Three questions, one method family, one cell each:**
1. **Overall RMSE** — are FCN3/GraphCast/HRES significantly different, pooling
   2016-2022 and all lead times?
2. **High-wind RMSE & Bias** — same question, restricted to obs >= 10.8 m/s.
3. **Year-by-year skill vs HRES** — is each model's relative skill stable over
   2016-2022, or drifting?



In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from IPython.display import display

OUT_ROOT      = Path('/cluster/work/projects/nn8106k/siyan/WF-experiments/case-study')
CSV_MASTER    = OUT_ROOT / 'master_forecast_dataset_72h_interp.csv'
CSV_HW_MERGED = OUT_ROOT / 'merged_fcn3_graphcast_hres_obs_all_years_72h_interp.csv'

YEARS         = [2016, 2017, 2018, 2019, 2020, 2021, 2022]
HIGH_WIND_THR = 10.8  # m/s -- 9-12 m/s bin (near-gale/gale, Beaufort 9-12)
MODEL_ORDER   = ['FCN3', 'GraphCast', 'HRES']
STATION_ORDER = ['SN88690', 'SN90490', 'SN90760', 'SN90800', 'SN90720', 'SN91740',
                  'SN89350', 'SN87110', 'SN85380', 'SN94500', 'SN96400', 'SN95350']

N_BOOT    = 10_000
BLOCK_LEN = 7  # days per resampled block (primary bootstrap)

print(f'YEARS={YEARS}  HIGH_WIND_THR={HIGH_WIND_THR} m/s  N_BOOT={N_BOOT:,}  BLOCK_LEN={BLOCK_LEN}d')

YEARS=[2016, 2017, 2018, 2019, 2020, 2021, 2022]  HIGH_WIND_THR=10.8 m/s  N_BOOT=10,000  BLOCK_LEN=7d


## Load Data

Both files are the same caches `rebuttal_model_comparison_analysis_72h_interpolate.ipynb`
builds (Sections 4 and 13) — read directly rather than rebuilt.


In [2]:
master_df = pd.read_csv(CSV_MASTER, parse_dates=['valid_time'])
master_long = master_df[['valid_time', 'lead_h', 'station', 'model', 'forecast', 'obs', 'year']]
print(f'master_df: {len(master_df):,} rows, models={sorted(master_df.model.unique())}, '
      f'years={sorted(master_df.year.unique())}')

merged_all = pd.read_csv(CSV_HW_MERGED, parse_dates=['valid_time'])
MODEL_COLS = [('fcn3_wind', 'FCN3'), ('graphcast_wind', 'GraphCast'), ('hres_wind', 'HRES')]
hw_long = pd.concat([
    merged_all.loc[merged_all['obs_wind'] >= HIGH_WIND_THR,
                    ['station', 'valid_time', 'lead_h', wind_col, 'obs_wind']]
        .rename(columns={wind_col: 'forecast', 'obs_wind': 'obs'})
        .assign(model=model)
    for wind_col, model in MODEL_COLS
], ignore_index=True)
print(f'High-wind subset (obs >= {HIGH_WIND_THR} m/s): {len(hw_long):,} rows')


master_df: 1,104,624 rows, models=['FCN3', 'GraphCast', 'HRES'], years=[np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
High-wind subset (obs >= 10.8 m/s): 120,741 rows


## Shared Machinery


In [3]:
def make_cell_stats(df_long):
    """(date, station, lead_h, model) -> n, sum_err, sum_sq_err."""
    df = df_long.dropna(subset=['forecast', 'obs']).copy()
    df['date'] = pd.to_datetime(df['valid_time']).dt.date
    df['err'] = df['forecast'] - df['obs']
    df['sq_err'] = df['err'] ** 2
    return (df.groupby(['date', 'station', 'lead_h', 'model'])
              .agg(n=('err', 'size'), sum_err=('err', 'sum'), sum_sq_err=('sq_err', 'sum'))
              .reset_index())


def dense_arrays(stats, model, dates, station_order=STATION_ORDER):
    """Dense (date, station, lead_h) arrays of n / sum_err / sum_sq_err."""
    leads = sorted(stats['lead_h'].unique())
    date_idx = {d: i for i, d in enumerate(dates)}
    station_idx = {s: j for j, s in enumerate(station_order)}
    lead_idx = {l: k for k, l in enumerate(leads)}
    sub = stats[stats['model'] == model]
    out = {k: np.zeros((len(dates), len(station_order), len(leads)))
           for k in ('n', 'sum_err', 'sum_sq_err')}
    di = sub['date'].map(date_idx).to_numpy()
    sj = sub['station'].map(station_idx).to_numpy()
    lk = sub['lead_h'].map(lead_idx).to_numpy()
    for k in out:
        out[k][di, sj, lk] = sub[k].to_numpy()
    return out


def metric_from_sums(sums, metric):
    n = sums['n']
    with np.errstate(invalid='ignore', divide='ignore'):
        if metric == 'RMSE':
            return np.sqrt(sums['sum_sq_err'] / n)
        if metric == 'Bias':
            return sums['sum_err'] / n
    raise ValueError(metric)


def station_lead_mean(sums, metric):
    """Metric per (station, lead_h), averaged over lead_h and then over stations --
    the same aggregation as the station-level cluster bootstrap (not pooled)."""
    m = metric_from_sums(sums, metric)
    return np.nanmean(np.nanmean(m, axis=-1), axis=-1)


def block_bootstrap_diff(arrays_a, arrays_b, metric, n_dates, block_len=BLOCK_LEN,
                          n_boot=N_BOOT, seed=0, chunk=250):
    """Primary test: moving block bootstrap 95% CI on mean_a(metric) - mean_b(metric).
    n_blocks = ceil(n_dates/block_len) start days are drawn with replacement, each
    expanded to `block_len` consecutive days (circular wrap)."""
    rng = np.random.default_rng(seed)
    n_blocks = int(np.ceil(n_dates / block_len))
    starts = rng.integers(0, n_dates, size=(n_boot, n_blocks))
    wrap = (np.arange(n_dates)[:, None] + np.arange(block_len)) % n_dates
    win_a = {k: v[wrap].sum(axis=1) for k, v in arrays_a.items()}   # block sums per start
    win_b = {k: v[wrap].sum(axis=1) for k, v in arrays_b.items()}
    diff = np.empty(n_boot)
    for lo in range(0, n_boot, chunk):
        st = starts[lo:lo + chunk]
        boot_a = {k: v[st].sum(axis=1) for k, v in win_a.items()}
        boot_b = {k: v[st].sum(axis=1) for k, v in win_b.items()}
        diff[lo:lo + chunk] = (station_lead_mean(boot_a, metric)
                                - station_lead_mean(boot_b, metric))

    obs_a = {k: v.sum(axis=0) for k, v in arrays_a.items()}
    obs_b = {k: v.sum(axis=0) for k, v in arrays_b.items()}
    obs_diff = float(station_lead_mean(obs_a, metric) - station_lead_mean(obs_b, metric))

    ci_low, ci_high = np.nanpercentile(diff, [2.5, 97.5])
    p = float(min(1.0, 2 * min(np.nanmean(diff <= 0), np.nanmean(diff >= 0))))
    return obs_diff, ci_low, ci_high, p


## Question 1 — Overall RMSE, All Years & Lead Times Pooled (vs Station Obs)

**Reviewer 1:** *"The paper uses a station-level cluster bootstrap with 10,000
resamples and only 12 stations. The number of bootstrap iterations is large, however,
the effective number of spatial clusters remains only 12, whereas the forecast errors
also have strong temporal correlation and shared weather-event dependence across
stations."*



In [4]:
stats_overall = make_cell_stats(master_long)
dates_overall = np.array(sorted(stats_overall['date'].unique()))
n_dates_overall = len(dates_overall)
arrays_overall = {m: dense_arrays(stats_overall, m, dates_overall) for m in MODEL_ORDER}

print(f'{n_dates_overall} calendar days\n')

q1_rows = []
for i, (m1, m2) in enumerate(combinations(MODEL_ORDER, 2)):
    obs, ci_low, ci_high, p_boot = block_bootstrap_diff(
        arrays_overall[m1], arrays_overall[m2], 'RMSE', n_dates_overall, seed=1000 + i)
    sig_boot = bool(ci_low > 0 or ci_high < 0)
    q1_rows.append({'model_a': m1, 'model_b': m2, 'mean_rmse_diff': obs,
                     'block_boot_ci_low': ci_low, 'block_boot_ci_high': ci_high,
                     'block_boot_p': p_boot, 'block_boot_significant': sig_boot})
    print(f'{m1} - {m2}: {obs:+.4f} m/s   {BLOCK_LEN}-day block-bootstrap 95% CI '
          f'[{ci_low:+.4f}, {ci_high:+.4f}]  p={p_boot:.4g} ({"sig" if sig_boot else "n.s."})')

q1_summary = pd.DataFrame(q1_rows)
display(q1_summary.round(4))


2557 calendar days

FCN3 - GraphCast: +0.0261 m/s   7-day block-bootstrap 95% CI [+0.0142, +0.0379]  p=0 (sig)
FCN3 - HRES: +0.0709 m/s   7-day block-bootstrap 95% CI [+0.0549, +0.0862]  p=0 (sig)
GraphCast - HRES: +0.0447 m/s   7-day block-bootstrap 95% CI [+0.0311, +0.0579]  p=0 (sig)


,model_a,model_b,mean_rmse_diff,block_boot_ci_low,block_boot_ci_high,block_boot_p,block_boot_significant
0,FCN3,GraphCast,0.0261,0.0142,0.0379,0.0,True
1,FCN3,HRES,0.0709,0.0549,0.0862,0.0,True
2,GraphCast,HRES,0.0447,0.0311,0.0579,0.0,True


## Question 2 — High-Wind RMSE & Bias (obs >= 10.8 m/s)


In [5]:
stats_hw = make_cell_stats(hw_long)
dates_hw = np.array(sorted(stats_hw['date'].unique()))
n_dates_hw = len(dates_hw)
arrays_hw = {m: dense_arrays(stats_hw, m, dates_hw) for m in MODEL_ORDER}

print(f'{n_dates_hw} high-wind calendar days\n')

q2_rows = []
for mi, metric in enumerate(['RMSE', 'Bias']):
    print(f'--- {metric} ---')
    for i, (m1, m2) in enumerate(combinations(MODEL_ORDER, 2)):
        obs, ci_low, ci_high, p_boot = block_bootstrap_diff(
            arrays_hw[m1], arrays_hw[m2], metric, n_dates_hw, seed=2000 + 10 * mi + i)
        sig_boot = bool(ci_low > 0 or ci_high < 0)
        q2_rows.append({'metric': metric, 'model_a': m1, 'model_b': m2, 'mean_diff': obs,
                         'block_boot_ci_low': ci_low, 'block_boot_ci_high': ci_high,
                         'block_boot_p': p_boot, 'block_boot_significant': sig_boot})
        print(f'  {m1} - {m2}: {obs:+.4f}   {BLOCK_LEN}-day block-bootstrap 95% CI '
              f'[{ci_low:+.4f}, {ci_high:+.4f}]  p={p_boot:.4g} ({"sig" if sig_boot else "n.s."})')

q2_summary = pd.DataFrame(q2_rows)
display(q2_summary.round(4))


1912 high-wind calendar days

--- RMSE ---
  FCN3 - GraphCast: -0.2436   7-day block-bootstrap 95% CI [-0.2876, -0.1993]  p=0 (sig)
  FCN3 - HRES: -0.2851   7-day block-bootstrap 95% CI [-0.3414, -0.2191]  p=0 (sig)
  GraphCast - HRES: -0.0415   7-day block-bootstrap 95% CI [-0.0962, +0.0178]  p=0.1608 (n.s.)
--- Bias ---
  FCN3 - GraphCast: +0.4476   7-day block-bootstrap 95% CI [+0.4012, +0.4934]  p=0 (sig)
  FCN3 - HRES: +0.5162   7-day block-bootstrap 95% CI [+0.4483, +0.5743]  p=0 (sig)
  GraphCast - HRES: +0.0686   7-day block-bootstrap 95% CI [+0.0069, +0.1266]  p=0.0308 (sig)


,metric,model_a,model_b,mean_diff,block_boot_ci_low,block_boot_ci_high,block_boot_p,block_boot_significant
0,RMSE,FCN3,GraphCast,-0.2436,-0.2876,-0.1993,0.0000,True
1,RMSE,FCN3,HRES,-0.2851,-0.3414,-0.2191,0.0000,True
2,RMSE,GraphCast,HRES,-0.0415,-0.0962,0.0178,0.1608,False
3,Bias,FCN3,GraphCast,0.4476,0.4012,0.4934,0.0000,True
4,Bias,FCN3,HRES,0.5162,0.4483,0.5743,0.0000,True
5,Bias,GraphCast,HRES,0.0686,0.0069,0.1266,0.0308,True


## Question 3 — Year-by-Year Relative Skill vs HRES (Temporal Generalization)

**Reviewer:** *"Could the authors report confidence intervals for the difference
between training-period and post-training-period performance, or yearly relative
skill against HRES? Such an analysis would better separate temporal distribution
shift from changes in annual forecast difficulty."*



In [6]:
q3_rows = []
for mi, model in enumerate(['FCN3', 'GraphCast']):
    for i, yr in enumerate(YEARS):
        df_year = master_long[master_long['year'] == yr]
        stats_year = make_cell_stats(df_year)
        dates_year = np.array(sorted(stats_year['date'].unique()))
        arrays_m = dense_arrays(stats_year, model, dates_year)
        arrays_h = dense_arrays(stats_year, 'HRES', dates_year)

        obs_diff, ci_low, ci_high, p = block_bootstrap_diff(
            arrays_m, arrays_h, 'RMSE', len(dates_year), seed=3000 + 100 * mi + i)
        sig = bool(ci_low > 0 or ci_high < 0)
        q3_rows.append({'model': model, 'year': yr, 'rmse_diff_vs_hres': obs_diff,
                         'ci_low': ci_low, 'ci_high': ci_high, 'p_value': p,
                         'significant_0.05': sig})

q3_summary = pd.DataFrame(q3_rows)
print(f'Yearly RMSE difference vs HRES (RMSE_model - RMSE_HRES), {BLOCK_LEN}-day block '
      'bootstrap 95% CI (positive = model worse than HRES that year):')
display(q3_summary.round(4))


Yearly RMSE difference vs HRES (RMSE_model - RMSE_HRES), 7-day block bootstrap 95% CI (positive = model worse than HRES that year):


,model,year,rmse_diff_vs_hres,ci_low,ci_high,p_value,significant_0.05
0,FCN3,2016,0.0231,-0.0141,0.0599,0.2212,False
1,FCN3,2017,0.0324,-0.0208,0.0815,0.2138,False
2,FCN3,2018,0.1087,0.0794,0.1380,0.0000,True
3,FCN3,2019,0.1062,0.0696,0.1419,0.0000,True
4,FCN3,2020,0.0733,0.0383,0.1082,0.0000,True
5,FCN3,2021,0.0632,0.0245,0.1009,0.0016,True
6,FCN3,2022,0.0894,0.0463,0.1322,0.0002,True
7,GraphCast,2016,0.0203,-0.0110,0.0492,0.1910,False
8,GraphCast,2017,0.0295,-0.0139,0.0699,0.1770,False
9,GraphCast,2018,0.0723,0.0485,0.0961,0.0000,True


## Sample Sizes per Setting

Data units per test, plus the bootstrap design: 10,000 draws, each resampling
ceil(days/7) blocks of 7 consecutive days (start points drawn with replacement).


In [7]:
def sample_sizes(df_long, label):
    d = df_long.dropna(subset=['forecast', 'obs'])
    st = make_cell_stats(d)
    dates = np.array(sorted(st['date'].unique()))
    per_day = st[st['model'] == st['model'].iloc[0]].groupby('date')['n'].sum() \
                .reindex(dates, fill_value=0).to_numpy()
    blk = per_day[(np.arange(len(dates))[:, None] + np.arange(BLOCK_LEN)) % len(dates)].sum(axis=1)
    return {'setting': label,
            'samples_per_block_mean': round(blk.mean()),
            'samples_per_block_min': int(blk.min()),
            'samples_per_block_max': int(blk.max()),
            'station_hours_per_model': len(d) // d['model'].nunique(),
            'station_days': st[['date', 'station']].drop_duplicates().shape[0],
            'calendar_days': len(dates),
            'stations': d['station'].nunique(),
            'bootstrap_draws': N_BOOT,
            'candidate_block_starts': len(dates),
            f'blocks_per_draw ({BLOCK_LEN}d each)': int(np.ceil(len(dates) / BLOCK_LEN)),
            'days_per_draw': int(np.ceil(len(dates) / BLOCK_LEN)) * BLOCK_LEN}

n_rows = [sample_sizes(master_long, 'Q1 overall 2016-2022'),
          sample_sizes(hw_long, f'Q2 high-wind (obs >= {HIGH_WIND_THR})')]
n_rows += [sample_sizes(master_long[master_long['year'] == y], f'Q3 year {y}') for y in YEARS]
display(pd.DataFrame(n_rows).set_index('setting'))


,samples_per_block_mean,samples_per_block_min,samples_per_block_max,station_hours_per_model,station_days,calendar_days,stations,bootstrap_draws,candidate_block_starts,blocks_per_draw (7d each),days_per_draw
setting,,,,,,,,,,,
Q1 overall 2016-2022,1004,813,1008,366585,30671,2557,12,10000,2557,366,2562
Q2 high-wind (obs >= 10.8),147,24,477,40247,7339,1912,12,10000,1912,274,1918
Q3 year 2016,1004,813,1008,52494,4390,366,12,10000,366,53,371
Q3 year 2017,1004,822,1008,52347,4379,365,12,10000,365,53,371
Q3 year 2018,1002,828,1008,52251,4376,365,12,10000,365,53,371
Q3 year 2019,1002,828,1008,52248,4375,365,12,10000,365,53,371
Q3 year 2020,1005,828,1008,52524,4392,366,12,10000,366,53,371
Q3 year 2021,1004,828,1008,52371,4380,365,12,10000,365,53,371
Q3 year 2022,1004,828,1008,52350,4379,365,12,10000,365,53,371
